# COP · Estudio académico reproducible

Este notebook ejecuta el estudio sobre **train** y usa **validation** para la comparación final. El test permanece cerrado.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

study_dir = Path(
    subprocess.check_output(
        [sys.executable, "scripts/run_academic_study.py"], text=True
    ).strip()
)
results = json.loads((study_dir / "study-results.json").read_text())
results["study_id"], results["test_status"]

In [ ]:
rows = []
for name, value in results["models"].items():
    validation = value["validation"]
    aggregate = value.get("cross_validation", {}).get("aggregate", {})
    rows.append(
        {
            "model": name,
            "validation_roc_auc": validation["roc_auc"],
            "validation_average_precision": validation["average_precision"],
            "validation_brier": validation["brier_score"],
            "cv_roc_auc_mean": aggregate.get("roc_auc", {}).get("mean"),
            "cv_roc_auc_std": aggregate.get("roc_auc", {}).get("std"),
        }
    )
metrics = pd.DataFrame(rows).set_index("model")
metrics

In [ ]:
threshold = pd.read_csv(study_dir / "threshold-sweep.csv")
model = "lightgbm"
frame = threshold[threshold["model"] == model].melt(
    id_vars="threshold",
    value_vars=["precision", "recall", "f1"],
    var_name="metric",
    value_name="value",
)
fig = px.bar(
    frame,
    x="metric",
    y="value",
    color="metric",
    animation_frame="threshold",
    range_y=[0, 1],
    title="Threshold lab",
)
fig.show()

## Lectura académica

- La validación cruzada mide estabilidad en train.
- Validation permite comparar la versión calibrada con el baseline.
- El threshold modifica la decisión descriptiva, no las probabilidades del modelo.
- No se selecciona champion ni se hacen claims de producción.